# 🩺 Chest Pathologies Diagnosis - Inference Notebook
This notebook loads the trained DenseNet model and performs inference on medical images.

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications.densenet import DenseNet121, preprocess_input
from tensorflow.keras.preprocessing import image
import os


In [5]:

labels = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
num_classes = len(labels)
print("Loaded class labels:", labels)


Loaded class labels: ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']


In [6]:

# Path to your trained weights
weights_path = r"D:\rehan\Projoects\Chest_pathalogies_diagnosis\23-aug-wieghts\23-aug-wieghts\saved_model.05-2.07.h5"

# Build model same as training
base_model = DenseNet121(weights=None, include_top=False)  # weights=None, since we'll load our own trained weights
x = base_model.output
x = GlobalAveragePooling2D()(x)
predictions = Dense(num_classes, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.load_weights(weights_path)

print(" Model loaded successfully with trained weights")


 Model loaded successfully with trained weights


In [7]:

def preprocess_image(img_path, target_size=(320,320)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img).astype("float32")

    # Standardize with training mean and std
    mean = 128.73689609375
    std = 74.05571891276443
    img_array = (img_array - mean) / std

    img_array = np.expand_dims(img_array, axis=0)  # add batch dimension
    return img_array

# Example usage
test_image_path = r"D:\rehan\Projoects\Chest_pathalogies_diagnosis\test_image.jpg"
img_array = preprocess_image(test_image_path, target_size=(320,320))


In [25]:
labels_to_predict = ['Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumothorax', 'Pleural Effusion']


In [30]:

pred = model.predict(img_array)[0]
pred


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 592ms/step


array([0.09334459, 0.18341894, 0.8123667 , 0.40676737, 0.25101474,
       0.87621075, 0.55111945, 0.60294425, 0.42442998, 0.06463742,
       0.72714436, 0.11106277, 0.13145205, 0.52878356], dtype=float32)

In [33]:

# --- Configuration File Path ---
CONFIG_FILE_PATH = 'inference_threshold_config.csv'

# 1. Load the saved configuration
try:
    inference_config = pd.read_csv(CONFIG_FILE_PATH, index_col='Class')
except FileNotFoundError:
    print(f"ERROR: Configuration file not found at {CONFIG_FILE_PATH}. Ensure the file is saved and the path is correct.")
    # Stop here or use a default
    # You must fix the file path if this error occurs
    
# Extract the necessary lists/arrays
inference_labels = inference_config.index.tolist()
inference_thresholds = inference_config['Threshold'].values

# --- FIX START: Ensure 'pred' is 2D ---
if pred.ndim == 1:
    # Reshape (14,) to (1, 14) so it can be indexed correctly
    pred = pred.reshape(1, -1)
    print("WARNING: 'pred' was 1-dimensional and has been reshaped to (1, 14).")
# --- FIX END ---


# 2. Map selected labels back to their original model index
# This links the selected feature name back to its column index in the 14-column 'pred' array
label_to_index = {label: i for i, label in enumerate(labels)}
try:
    inference_indices = [label_to_index[label] for label in inference_labels]
except KeyError as e:
    print(f"ERROR: Label {e} found in config file but not in original class_labels. Check consistency.")
    raise

# 3. Select the probability columns for only the high-performing features
selected_pred_probs = pred[:, inference_indices]

# 4. Apply the feature-specific best thresholds
# Reshape thresholds for correct NumPy broadcasting (1, N_features)
thresholds_reshaped = inference_thresholds.reshape(1, -1)

# Apply thresholding: convert probabilities to binary (0 or 1) predictions
binary_predictions = (selected_pred_probs > thresholds_reshaped).astype(int)


# --- Final Output ---
print(f"Successfully generated binary predictions for {len(inference_labels)} features.")

# Optional: Display the results nicely
df_final_predictions = pd.DataFrame(
    binary_predictions,
    columns=inference_labels
)
print("\nFinal Binary Predictions (First 5 samples):")
df_final_predictions.head()

Successfully generated binary predictions for 6 features.

Final Binary Predictions (First 5 samples):


,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumothorax,Pleural Effusion
0,1,0,1,1,0,1
